# Cell 1: 纯数据损失（无物理约束）— 快速验证网络能拟合背景

In [ ]:
import torch
from torch import optim
from tqdm.notebook import tqdm

import pinn_starlight_core.nn.Layers as Layers
import pinn_starlight_core.nn.Losses as Loss
import pinn_starlight_core.data.RAWLoader as RAWLoader
import pinn_starlight_core.data.FakeRAW as FakeRAW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

raw_loader = RAWLoader.RAWLoader()
raw_loader.from_array(FakeRAW.FakeRaw(W=256, H=256).get_fake_raw())
coords, values, W, H = raw_loader.get_raw_data(device=device)

layers = [
    Layers.SkyglowLinear(2, 256).to(device), Layers.SkyglowActivation(),
    Layers.SkyglowLinear(256, 64).to(device), Layers.SkyglowActivation(),
    Layers.SkyglowLinear(64, 1).to(device),
]
params = [p for l in layers if isinstance(l, Layers.SkyglowLinear) for p in l.parameters()]
optimizer = optim.Adam(params, lr=0.001)
loss_fn = Loss.MSEData()

N = coords.shape[0]
batch = max(1024, N // 200)

for step in tqdm(range(2000), desc='Data-only'):
    idx = torch.randint(0, N, (batch,), device=device)
    a = coords[idx]
    for l in layers: a = l.forward(a)
    loss = loss_fn.forward(values[idx], a.squeeze())
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 500 == 0:
        print(f'Step {step}: Loss={loss.item():.6f}')
print('Done.')

# Cell 2: 真实图片批量处理（物理损失）

In [ ]:
# (此 cell 已被清理 — 训练用 Cell 2 即可)

# Cell 3: 出图

In [1]:
from pinn_starlight_core.nn import Icity
import os
import torch
from torch import optim
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import pinn_starlight_core.nn.Layers as Layers
import pinn_starlight_core.nn.Losses as Loss
import pinn_starlight_core.data.RAWLoader as RAWLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_dir  = '../../data/real_raw/origin'
output_dir = '../../data/real_raw/trained'
os.makedirs(output_dir, exist_ok=True)

for file in sorted(os.listdir(input_dir)):
    path = os.path.join(input_dir, file)
    base = file.rsplit('.', 1)[0]
    print(f'Processing {file}...')

    loader = RAWLoader.RAWLoader()
    loader.load(path)
    coords, values, W, H = loader.get_raw_data(device)

    layers = [
        Layers.SkyglowLinear(2, 512).to(device),
        Layers.SkyglowActivation(),
        Layers.SkyglowLinear(512, 64).to(device),
        Layers.SkyglowActivation(),
        Layers.SkyglowLinear(64, 1).to(device),
    ]

    params = []
    for layer in layers:
        if isinstance(layer, Layers.SkyglowLinear):
            params += list(layer.parameters())

    ld = Loss.MSEData()
    lp = Loss.MSEPhysics()

    alpha = torch.nn.Parameter(torch.tensor(1.0, device=device))
    I_city = Icity.LearnableIcity(values=values, device=device)
    phy_weight = 0.1
    tv_weight = 0.01

    optimizer = optim.Adam(
        params +
        list(I_city.parameters()) +
        [alpha],
        lr = 0.001
    )

    for step in tqdm(range(int(coords.shape[0] / 330))):
        idx = torch.randint(0, coords.shape[0], (int(coords.shape[0] / 420),))
        batch_xy = coords[idx].to(device).clone().requires_grad_(True)
        batch_I = values[idx].to(device)

        a = batch_xy
        for layer in layers:
            a = layer.forward(a)
        I_pred = a.squeeze().to(device)

        I_city_vals = I_city(batch_xy)

        data_loss = ld.forward(batch_I, I_pred)
        phys_loss = lp.forward(batch_I, I_pred, I_city_vals, alpha, batch_xy)
        tv_loss = Icity.tv_loss(I_city.grid)
        loss = data_loss + phy_weight * phys_loss + tv_weight * tv_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        I_pred = torch.empty(coords.shape[0])
        for start in range(0, coords.shape[0], 50000):
            end = min(start + 50000, coords.shape[0])
            a = coords[start:end]
            for layer in layers:
                a = layer.forward(a)
            I_pred[start:end] = a.squeeze()

    obs = values.reshape(W, H).cpu().numpy()
    pred = I_pred.reshape(W, H).cpu().numpy()
    res = (obs - pred).clip(0, 1)

    print(f'alpha:{alpha},')
    plt.imsave(f'{output_dir}/{base}_observed.png', obs, cmap='gray')
    plt.imsave(f'{output_dir}/{base}_predicted.png', pred, cmap='gray')
    plt.imsave(f'{output_dir}/{base}_residual.png', res, cmap='gray')

print('Done.')

Processing 01.jpg...


  0%|          | 0/360 [00:00<?, ?it/s]

alpha:Parameter containing:
tensor(0.9473, requires_grad=True),
Processing 02.jpg...


  0%|          | 0/3958 [00:00<?, ?it/s]

alpha:Parameter containing:
tensor(-1.6112, requires_grad=True),
Processing 03.jpg...


  0%|          | 0/1045 [00:00<?, ?it/s]

alpha:Parameter containing:
tensor(0.3480, requires_grad=True),
Processing 04.jpg...


  0%|          | 0/9630 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Cell 4: 合成数据评估（已知真值）

In [ ]:
# === Cell 4: 合成数据评估（已知真值，含物理损失）===
import torch
from torch import optim
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import pinn_starlight_core.nn.Layers as Layers
import pinn_starlight_core.nn.Losses as Loss
import pinn_starlight_core.data.RAWLoader as RAWLoader
import pinn_starlight_core.data.FakeRAW as FakeRAW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

fake = FakeRAW.FakeRaw(W=256, H=256, n_stars=12, bg_amplitude=0.3, star_brightness=0.7, seed=42)
loader = RAWLoader.RAWLoader()
loader.from_array(fake.get_fake_raw())
coords, values, W, H = loader.get_raw_data(device=device)

true_bg = fake.background.flatten().to(device)
true_stars = fake.stars.flatten().to(device)

layers = [
    Layers.SkyglowLinear(2, 256).to(device), Layers.SkyglowActivation(),
    Layers.SkyglowLinear(256, 64).to(device), Layers.SkyglowActivation(),
    Layers.SkyglowLinear(64, 1).to(device),
]
params = [p for l in layers if isinstance(l, Layers.SkyglowLinear) for p in l.parameters()]
optimizer = optim.Adam(params, lr=0.001)
ld, lp = Loss.MSEData(), Loss.MSEPhysics()

# 指数背景: bg = A*exp(-(x+y)/D), ∇²bg = 2bg/D²
alpha, D_bg = 4.0, 0.7
bg = 0.3 * torch.exp(-(coords[:, 0] + coords[:, 1]) / D_bg).to(device)
I_city = (alpha - 2.0 / D_bg**2) * bg
phy_weight = 0.01

N = coords.shape[0]
batch = max(1024, N // 200)

for step in tqdm(range(3000), desc='Training'):
    idx = torch.randint(0, N, (batch,), device=device)
    xy = coords[idx].clone().requires_grad_(True)
    a = xy
    for l in layers: a = l.forward(a)
    Ip = a.squeeze()
    loss = ld.forward(values[idx], Ip) + lp.forward(values[idx], Ip, I_city[idx], alpha, phy_weight, xy)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

with torch.no_grad():
    I_pred = torch.empty(N, device=device)
    for s in range(0, N, 50000):
        e = min(s+50000, N); a = coords[s:e]
        for l in layers: a = l.forward(a)
        I_pred[s:e] = a.squeeze()

bg_mse = ((I_pred - true_bg)**2).mean().item()
star_res = (values - I_pred).clamp(0, 1)
star_recovered = star_res.sum().item() / true_stars.sum().item()

print(f'\n背景 MSE: {bg_mse:.6f}  |  星点回收率: {star_recovered:.2%}')

obs = values.reshape(W, H).cpu().numpy()
pred = I_pred.reshape(W, H).cpu().numpy()
res = star_res.reshape(W, H).cpu().numpy()
truth = true_bg.reshape(W, H).cpu().numpy()

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, arr, t in zip(ax, [obs, pred, res, truth],
    ['Observed', 'Predicted (bg)', 'Residual (stars)', 'True Background']):
    a.imshow(arr, cmap='gray'); a.set_title(t); a.axis('off')
fig.tight_layout(); plt.show()